# 📖 Next Word Predictor using LSTM

This project trains a Long Short-Term Memory (LSTM) neural network to predict the next word in a sequence of text, using **"The Adventures of Sherlock Holmes"** by Arthur Conan Doyle (public domain, via Project Gutenberg) as the training corpus.

**Pipeline:**
1. Download & clean the dataset
2. Tokenize the text and build the vocabulary
3. Generate n-gram training sequences
4. Pad sequences and one-hot encode the labels
5. Build and train a stacked LSTM model
6. Use the trained model to generate text, word by word

> Run this notebook top-to-bottom in **Google Colab**. For faster training, go to `Runtime > Change runtime type > GPU`.

## 1. Setup

Import the libraries we'll need for text processing, sequence handling, and building the LSTM model.

In [ ]:
import re
import urllib.request
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

print("TensorFlow version:", tf.__version__)

## 2. Download the Dataset

We use **"The Adventures of Sherlock Holmes"** (Project Gutenberg eBook #1661), a public-domain text with rich, varied vocabulary — much better suited to demonstrating an LSTM's language modeling ability than a short FAQ document.

In [ ]:
url = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"
raw_text = urllib.request.urlopen(url).read().decode("utf-8")

print("Total characters downloaded:", len(raw_text))
print(raw_text[:500])

## 3. Clean the Text

Project Gutenberg files include license/header/footer boilerplate that we don't want the model to learn from. We strip that out, then do some light cleaning:
- Remove the Gutenberg header and footer
- Remove chapter headings / all-caps title lines
- Collapse whitespace and remove characters that aren't useful for a word-level model
- Lowercase everything (keeps the vocabulary smaller and more learnable)

In [ ]:
def clean_gutenberg_text(text):
    # Strip Project Gutenberg boilerplate
    start_match = re.search(r"\*\*\* START OF (THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.DOTALL)
    end_match = re.search(r"\*\*\* END OF (THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.DOTALL)
    if start_match:
        text = text[start_match.end():]
    if end_match:
        text = text[:end_match.start()]
    return text

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[_*]", "", text)              # remove markdown-ish emphasis chars
    text = re.sub(r"[^a-z0-9.,!?;:'\n ]", " ", text)  # keep basic punctuation only
    text = re.sub(r"[ \t]+", " ", text)            # collapse repeated spaces
    text = re.sub(r"\n{2,}", "\n", text)           # collapse repeated blank lines
    return text.strip()

book_text = clean_gutenberg_text(raw_text)
book_text = clean_text(book_text)

print("Cleaned length (characters):", len(book_text))
print(book_text[:500])

To keep training time reasonable on Colab's free tier, we'll use a meaningful excerpt rather than the entire book — the first few thousand lines are more than enough for the model to learn sentence structure and word patterns.

In [ ]:
lines = [line.strip() for line in book_text.split("\n") if len(line.strip().split()) > 3]
lines = lines[:4000]   # keep a substantial but manageable chunk

corpus_text = "\n".join(lines)
print("Number of lines used:", len(lines))
print("Total words (approx):", len(corpus_text.split()))

## 4. Tokenization

Just like the original project, we use Keras's `Tokenizer` to build a word-level vocabulary, mapping every unique word to an integer.

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus_text])

vocab_size = len(tokenizer.word_index) + 1   # +1 for padding token (index 0)
print("Vocabulary size:", vocab_size)

## 5. Generate N-gram Training Sequences

For every line, we create progressively longer sub-sequences. For example, the sentence `"to sherlock holmes she is always the woman"` becomes multiple training examples:

```
[to, sherlock]
[to, sherlock, holmes]
[to, sherlock, holmes, she]
...
```

Each sequence's last word becomes the label the model must learn to predict from everything before it.

In [ ]:
input_sequences = []

for sentence in corpus_text.split("\n"):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(tokenized_sentence)):
        input_sequences.append(tokenized_sentence[:i + 1])

print("Total training sequences:", len(input_sequences))
print("Example sequence:", input_sequences[10])

## 6. Pad Sequences

Sequences have different lengths, but neural networks need fixed-size input. We left-pad (`padding='pre'`) every sequence up to the length of the longest one, so the word to predict always sits at the very end.

In [ ]:
max_len = max(len(seq) for seq in input_sequences)
print("Max sequence length:", max_len)

padded_input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')
print("Padded shape:", padded_input_sequences.shape)

## 7. Create Predictors (X) and Labels (y)

The last column of every padded sequence is the word to predict (`y`); everything before it is the input context (`X`). We then one-hot encode `y` across the full vocabulary, since this is a multi-class classification problem (predict *which* word, out of the whole vocabulary, comes next).

In [ ]:
X = padded_input_sequences[:, :-1]
y = padded_input_sequences[:, -1]

y = to_categorical(y, num_classes=vocab_size)

print("X shape:", X.shape)
print("y shape:", y.shape)

## 8. Build the LSTM Model

Same core architecture as the original project — an `Embedding` layer followed by two stacked `LSTM` layers and a `Dense` softmax output over the vocabulary. Since this vocabulary is much larger than the original 283-word example, we add light `Dropout` for regularization to help prevent overfitting.

In [ ]:
model = Sequential()
model.add(Embedding(vocab_size, 100, input_length=max_len - 1))
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(150))
model.add(Dropout(0.2))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

## 9. Train the Model

Training a word-level language model takes a while — start with a modest number of epochs and increase if you have GPU time to spare. Watch the `accuracy` metric: it should climb steadily as the model learns common word patterns from Victorian-era English.

In [ ]:
history = model.fit(X, y, epochs=50, batch_size=128, verbose=1)

Optional: plot training accuracy/loss to see how learning progressed.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'])
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')

axes[1].plot(history.history['accuracy'])
axes[1].set_title('Training Accuracy')
axes[1].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

## 10. Save the Model

Save the trained model and tokenizer so they can be reloaded later without retraining.

In [ ]:
model.save("next_word_lstm.keras")

import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Model and tokenizer saved.")

## 11. Generate Text with the Trained Model

Given a starting phrase, we repeatedly:
1. Tokenize and pad the current text
2. Predict the most likely next word
3. Append it to the text and repeat

In [ ]:
def predict_next_words(seed_text, num_words=10):
    for _ in range(num_words):
        token_text = tokenizer.texts_to_sequences([seed_text])[0]
        padded_token_text = pad_sequences([token_text], maxlen=max_len - 1, padding='pre')
        predicted_index = np.argmax(model.predict(padded_token_text, verbose=0))

        next_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                next_word = word
                break

        seed_text = seed_text + " " + next_word

    return seed_text

print(predict_next_words("sherlock holmes was", num_words=15))

## Next Steps / Ideas to Extend This Project

- Train on the **full** book (or add more Sherlock Holmes stories) for richer predictions
- Try `GRU` layers instead of `LSTM` and compare performance
- Add **temperature-based sampling** instead of always picking the single most likely word (produces more varied, natural-sounding text)
- Wrap the trained model in a simple **Streamlit** or **Gradio** app for an interactive demo
- Track **perplexity** as a more standard language-modeling evaluation metric